In [ ]:
## Import necessary modules
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, matthews_corrcoef
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sn
import seaborn as sns
import pandas as pd
import csv
import math
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score,KFold,StratifiedKFold,cross_val_predict
from sklearn.metrics import classification_report
import sklearn
import tensorflow as tf
import keras
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.models import Model
from keras.callbacks import ReduceLROnPlateau
from keras.regularizers import l2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pdv
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Features/ACPs_QLC_Training.csv')

In [ ]:
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Replace inf values and handle missing values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(999, inplace=True)

# Split features and labels
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"Original data shape: {X.shape}")
print(f"Original class distribution: {Counter(y)}")

# =========================
# 1️⃣ Train-Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")
print(f"Training class distribution: {Counter(y_train)}")
print(f"Testing class distribution: {Counter(y_test)}")

# =========================
# 2️⃣ Standardization (Fit only on training data)
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
!pip install keras-tcn
from tcn import TCN

In [ ]:
from keras.layers import LayerNormalization
from sklearn.model_selection import train_test_split, StratifiedKFold
from keras.layers import Conv1D
from keras import layers, models
from keras.optimizers import Adam

def TCNBlock(x, filters, kernel_size, dilation_rate):
    x = layers.Conv1D(
        filters,
        kernel_size,
        padding='causal',
        dilation_rate=dilation_rate,
        activation='linear'
    )(x)
    x = layers.LayerNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.15)(x)
    return x
# -------------------------
# Parallel Multi-Scale TCN model
# -------------------------
def create_tcn_model(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Branch 1 (small receptive field)
    b1 = TCNBlock(inputs, filters=32, kernel_size=1, dilation_rate=1)
    b1 = TCNBlock(b1,     filters=32, kernel_size=1, dilation_rate=2)
    b1 = TCNBlock(b1,     filters=32, kernel_size=1, dilation_rate=4)

    # Branch 2 (medium receptive field)
    b2 = TCNBlock(inputs, filters=32, kernel_size=3, dilation_rate=1)
    b2 = TCNBlock(b2,     filters=32, kernel_size=3, dilation_rate=2)
    b2 = TCNBlock(b2,     filters=32, kernel_size=3, dilation_rate=4)

    # Branch 3 (large receptive field)
    b3 = TCNBlock(inputs, filters=32, kernel_size=5, dilation_rate=1)
    b3 = TCNBlock(b3,     filters=32, kernel_size=5, dilation_rate=2)
    b3 = TCNBlock(b3,     filters=32, kernel_size=5, dilation_rate=4)

    # ---- Fuse parallel branches ----
    x = layers.Concatenate(axis=-1)([b1, b2, b3])

    # Project back to 32 channels so residual matches
    x = layers.Conv1D(32, kernel_size=3, padding="same")(x)

    # Residual connection (match channels)
    res = layers.Conv1D(32, kernel_size=3, padding="same")(inputs)
    x = layers.Add()([x, res])

    # Final layers
    x = layers.Flatten()(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model

# Define input shape
input_shape = (X_train.shape[1], 1)
model = create_tcn_model(input_shape)

# Compile the model
learning_rate = 0.001
optimizer = Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Initialize lists to store evaluation metrics for each fold
accuracy_list = []
f1_list = []
precision_list = []
recall_list = []
sensitivity_list = []
specificity_list = []
mcc_list = []
auc_list = []

# Perform K-fold cross-validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
from keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 在K-fold循环中
for train_index, val_index in skf.split(X_train,y_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # 定义回调
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

    callbacks = [early_stop, reduce_lr]

    # 训练
    history = model.fit(X_train_fold, y_train_fold,
                       epochs=80,
                       batch_size=32,
                       verbose=0,
                       #validation_split=0.1,
                       validation_data=(X_val_fold, y_val_fold),
                       callbacks=callbacks)

    # 保持您原有的评估代码不变
    y_val_pred = model.predict(X_val_fold)
    y_val_pred_binary = (y_val_pred > 0.5).astype(int)

    # Calculate confusion matrix for the current fold
    cm = confusion_matrix(y_val_fold, y_val_pred_binary)
    # Calculate AUC for the current fold
    auc = roc_auc_score(y_val_fold, y_val_pred)
    auc_list.append(auc)


    # Calculate performance evaluation metrics for the current fold
    TN = cm[0, 0]
    FP = cm[0, 1]
    FN = cm[1, 0]
    TP = cm[1, 1]

    # Accuracy
    accuracy = (TP + TN) / float(TP + TN + FP + FN)
    accuracy_list.append(accuracy)

    # F1-Score
    f1 = 2 * TP / float(2 * TP + FP + FN)
    f1_list.append(f1)

    # Precision
    precision = TP / float(TP + FP)
    precision_list.append(precision)

    # Recall
    recall = TP / float(TP + FN)
    recall_list.append(recall)

    # Sensitivity
    sensitivity = TP / float(TP + FN)
    sensitivity_list.append(sensitivity)

    # Specificity
    specificity = TN / float(TN + FP)
    specificity_list.append(specificity)

    # MCC (Matthews Correlation Coefficient)
    mcc = ((TP * TN) - (FP * FN)) / float((np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))) or 1)
    mcc_list.append(mcc)

# Calculate average performance evaluation metrics across all folds
avg_accuracy = np.mean(accuracy_list)
avg_f1 = np.mean(f1_list)
avg_precision = np.mean(precision_list)
avg_recall = np.mean(recall_list)
avg_sensitivity = np.mean(sensitivity_list)
avg_specificity = np.mean(specificity_list)
avg_mcc = np.mean(mcc_list)

# Calculate average AUC across all folds
avg_auc = np.mean(auc_list)
# Print average performance evaluation metrics
print("Accuracy =", avg_accuracy)
print("F1 Score =", avg_f1)
print("Precision =", avg_precision)
print("Recall =", avg_recall)
print("Sensitivity =", avg_sensitivity)
print("Specificity =", avg_specificity)
print("MCC =", avg_mcc)
print("AUC =", avg_auc)